# w5_cpu_dair_attached_o2_bioeq


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK, MOUNT = Path("/kaggle/working"), Path("/kaggle/input")
assert WORK.is_dir(), "Run on Kaggle"
REV = "c94b531b7e7a28ceea6f8218fb35613697e0eaec"
with tempfile.TemporaryDirectory(prefix="w5_dair_") as ts:
    tmp = Path(ts)
    repo, meta_dir = tmp / "repo", tmp / "pinns_meta"
    subprocess.run(["git", "init", "-q", str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "fetch", "-q", "--depth=1", "https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git", REV], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    env = {**os.environ, "PYTHONPATH": str(repo), "PYTHONDONTWRITEBYTECODE": "1", "CUDA_VISIBLE_DEVICES": ""}
    heldout_bin = next(MOUNT.rglob("shortpose_heldout.bin"))
    dair_zip = next(MOUNT.rglob("dair_roadside_third_cohort.zip"))
    pose_npz = tmp / "pose.npz"
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read("pose.npz"))
    subprocess.run([sys.executable, "-m", "scripts.stage_pinns_metadata", "--output-dir", str(meta_dir)], cwd=repo, env=env, check=True)
    dest = WORK / "dair_o2_bioeq_external.zip"
    subprocess.run([sys.executable, "-u", "-m", "scripts.run_dair_attached_o2_bioeq", str(meta_dir), str(pose_npz), str(dair_zip), str(dest)], cwd=repo, env=env, check=True)
    print("DONE:", dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
